## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [2]:
effect_size = sms.proportion_effectsize(0.2, 0.19)
print(f'На основі наших очікуваних показників розмір ефекту  - {(effect_size)*100:.2f}%.')

На основі наших очікуваних показників розмір ефекту  - 2.52%.


In [3]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )                                               

required_n = ceil(required_n)                         

print(f'Аби досягнути такого ефекту, сумарно потрібно {(required_n)*2} користувачів')

Аби досягнути такого ефекту, сумарно потрібно 49276 користувачів


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [ ]:
df = pd.read_csv("..//data/statistical_hypothesis/cookie_cats.csv")
type(df)
df.info()

In [ ]:
retention = df.groupby("version")["retention_7"].mean()*100

diff = retention['gate_30'] - retention['gate_40']

print(f"gate_30: {retention['gate_30']:.2f}%")
print(f"gate_40: {retention['gate_40']:.2f}%")
print(f"Різниця: {diff:.2f}%")
print("Версія gate_30 демонструє дещо вище утримання користувачів на 7 день.")
print("Однак необхідно перевірити, чи є ця різниця статистично значущою.")



3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


Гіпотези:

$H_0: p_{30} = p_{40}$ — утримання на 7 день однакове.

$H_a: p_{30} \ne p_{40}$ — утримання на 7 день різне.

Тест: **двосторонній**.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control_results = df[df['version'] == 'gate_30']
treatment_results = df[df['version'] == 'gate_40']

successes = [control_results['retention_7'].sum(), treatment_results['retention_7'].sum()]
nobs = [len(control_results), len(treatment_results)]
z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat}')
print(f'p-value: {pval}')
if pval < 0.05:
    print('Відхиляємо гіпотезу H₀: є статистично значуща різниця в утриманні на 7 день між версіями гри.')
else:
    print('Не відхиляємо гіпотезу H₀: недостатньо доказів, що утримання на 7 день відрізняється між версіями.')
    
print(f'Довірчий інтервал 95% для групи "gate_30": [{lower_con:.3f}, {upper_con:.3f}].')
print(f'Довірчий інтервал 95% для групи "gate_40": [{lower_treat:.3f}, {upper_treat:.3f}].')
print('Довірчі інтервали для цих версій не перетинаються, що додатково підтверджує наявність значущої різниці.')
print('Версія "gate_30" демонструє вище утримання користувачів, тому вона є більш ефективною з точки зору утримання гравців.')

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Рішення:

Тест Хі-квадрат перевіряє чи залежить утримання на 7 день від версіїї гри, тому гіпотези будуть такі:

Гіпотези:


$H_0$: версія гри та утримання незалежні

$H_a$: версія гри впливає на утримання 


In [ ]:
from scipy.stats import chi2_contingency

alpha = 0.05
table = pd.crosstab(df['version'], df['retention_7'])


In [ ]:
table

In [ ]:
chi2, pval, dof, expected = chi2_contingency(table)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {pval:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected.round(2))
if pval < alpha:
    print('Відхиляємо гіпотезу H₀')
    print('Версія гри впливає на ймовірність того, що гравець повернеться через 7 днів.')
else:
    print('Не відхиляємо гіпотезу H₀')
    print('Версія гри НЕ впливає на ймовірність того, що гравець повернеться через 7 днів.')